# Stable Diffusion 1.5 Universal Generator (Improved)
This notebook has been upgraded to support:
- **All SD 1.5 Models**: Load from Hugging Face Repo ID or direct CivitAI download links.
- **Gradio UI**: Easy-to-use web interface.
- **Advanced Controls**: Negative prompts, Steps, CFG Scale, Seed.
- **Google Drive Integration**: Automatically save your creations to Drive.

In [ ]:
!pip install -q diffusers transformers accelerate safetensors gradio omegaconf invisible-watermark

In [ ]:
import torch
import gradio as gr
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
import os
import requests
from datetime import datetime
from google.colab import drive

# Global variable to hold the pipe so we don't reload it unnecessarily
pipe = None
current_model_path = ""
drive_mounted = False

def mount_google_drive():
    global drive_mounted
    if not drive_mounted:
        try:
            drive.mount('/content/drive')
            drive_mounted = True
            return True
        except Exception as e:
            print(f"Error mounting drive: {e}")
            return False
    return True

def load_model(model_path_or_url):
    global pipe, current_model_path
    
    # Avoid reloading if it's the same model
    if pipe is not None and model_path_or_url == current_model_path:
        return "Model already loaded."

    print(f"Loading model: {model_path_or_url}...")
    
    try:
        # Scenario 1: It's a Hugging Face Repo ID (e.g., "runwayml/stable-diffusion-v1-5")
        if not model_path_or_url.startswith("http") and "/" in model_path_or_url:
            pipe = StableDiffusionPipeline.from_pretrained(
                model_path_or_url, 
                torch_dtype=torch.float16,
                use_safetensors=True
            )
            
        # Scenario 2: It's a URL (CivitAI or Direct Link)
        else:
            filename = "custom_model.safetensors"
            
            # If it's a URL, we might need to download it first
            if model_path_or_url.startswith("http"):
                print("Downloading model...")
                # Basic download logic (supports CivitAI redirects)
                response = requests.get(model_path_or_url, stream=True)
                response.raise_for_status()
                
                # Try to get filename from content-disposition
                if "content-disposition" in response.headers:
                    import re
                    fname = re.findall("filename=(.+)", response.headers["content-disposition"])
                    if fname:
                        filename = fname[0].strip('"')
                
                with open(filename, "wb") as f:
                    for chunk in response.iter_content(chunk_size=8192):
                        f.write(chunk)
                print(f"Downloaded to {filename}")
                model_path_or_url = filename

            # Load from single file (Safetensors/CKPT)
            pipe = StableDiffusionPipeline.from_single_file(
                model_path_or_url,
                torch_dtype=torch.float16
            )

        # Optimization for Colab (SD 1.5)
        pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
        pipe.enable_model_cpu_offload() # Saves GPU VRAM
        pipe.safety_checker = None # Optional: Disable safety checker if preferred
        
        current_model_path = model_path_or_url
        return "Model loaded successfully!"
        
    except Exception as e:
        return f"Error loading model: {str(e)}"

def generate_image(prompt, negative_prompt, steps, cfg_scale, width, height, seed, num_images, save_to_drive, drive_folder):
    global pipe
    if pipe is None:
        return [None] * num_images # Handle error

    generator = torch.Generator(device="cpu").manual_seed(int(seed))
    
    images = pipe(
        prompt,
        negative_prompt=negative_prompt,
        num_inference_steps=steps,
        guidance_scale=cfg_scale,
        width=width,
        height=height,
        num_images_per_prompt=num_images,
        generator=generator
    ).images
    
    # Save images if requested
    if save_to_drive:
        if mount_google_drive():
            save_path = f"/content/drive/MyDrive/{drive_folder}"
            os.makedirs(save_path, exist_ok=True)
            
            for i, img in enumerate(images):
                timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                filename = f"{save_path}/sd_{timestamp}_{i}.png"
                img.save(filename)
                print(f"Saved to {filename}")
    
    return images

# --- Gradio UI Layout ---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# Stable Diffusion 1.5 Universal Generator")
    
    with gr.Row():
        with gr.Column(scale=1):
            # Model Selection
            model_input = gr.Textbox(
                label="Model URL (CivitAI) or HF Repo ID", 
                value="runwayml/stable-diffusion-v1-5",
                placeholder="e.g. https://civitai.com/api/download/models/12345 or runwayml/stable-diffusion-v1-5"
            )
            load_btn = gr.Button("Load Model", variant="secondary")
            status_text = gr.Textbox(label="Status", interactive=False)
            
            # Generation Params
            prompt = gr.Textbox(label="Prompt", lines=3, placeholder="Describe your image...")
            neg_prompt = gr.Textbox(label="Negative Prompt", lines=3, value="low quality, bad anatomy, worst quality")
            
            with gr.Accordion("Advanced Settings", open=True):
                with gr.Row():
                    width = gr.Slider(label="Width", minimum=256, maximum=1024, step=64, value=512)
                    height = gr.Slider(label="Height", minimum=256, maximum=1024, step=64, value=512)
                
                steps = gr.Slider(label="Steps", minimum=10, maximum=100, step=1, value=25)
                cfg = gr.Slider(label="Guidance Scale (CFG)", minimum=1, maximum=20, step=0.5, value=7.5)
                seed = gr.Number(label="Seed", value=42, precision=0)
                batch_size = gr.Slider(label="Number of Images", minimum=1, maximum=4, step=1, value=1)

            # Drive Settings
            with gr.Group():
                save_drive = gr.Checkbox(label="Save Images to Google Drive", value=False)
                drive_folder = gr.Textbox(label="Drive Folder Name", value="SD_Outputs")

            gen_btn = gr.Button("Generate", variant="primary")

        with gr.Column(scale=1):
            gallery = gr.Gallery(label="Generated Images")

    # Event Wiring
    load_btn.click(fn=load_model, inputs=[model_input], outputs=[status_text])
    gen_btn.click(
        fn=generate_image,
        inputs=[prompt, neg_prompt, steps, cfg, width, height, seed, batch_size, save_drive, drive_folder],
        outputs=[gallery]
    )

demo.launch(share=True, debug=True)